# **LLMs et LangChain**

<figure>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7HnZLgyttvmbXmXf0tl_FQ/201033-AdobeStock-1254756887%20571x367.png" 
</figure>

## Introduction

LangChain est un framework open-source conçu pour développer des applications exploitant des modèles de langage de grande taille (LLMs). LangChain se distingue en fournissant des outils et des abstractions essentiels qui améliorent la personnalisation, la précision et la pertinence des informations générées par ces modèles.

LangChain propose une interface générique compatible avec presque n’importe quel LLM. Cette interface facilite un environnement de développement centralisé permettant aux data scientists d’intégrer sans difficulté des applications basées sur les LLM avec des sources de données externes et des workflows logiciels. Cette intégration est essentielle pour les organisations souhaitant exploiter pleinement le potentiel de l’IA dans leurs processus.

L’une des fonctionnalités les plus puissantes de LangChain est son approche basée sur des modules. Cette approche favorise la flexibilité lors des expérimentations et l’optimisation des interactions avec les LLM. Les data scientists peuvent comparer dynamiquement les prompts et basculer entre différents modèles fondamentaux sans modifications majeures du code. Ces capacités permettent de gagner un temps de développement précieux et d’améliorer la capacité des développeurs à affiner les applications.


## 1. Objectifs

Après avoir terminé ce TP, vous serez en mesure de :

- Utiliser les fonctionnalités principales du framework LangChain, y compris les modèles de prompts, les chaînes (chains) et les agents, afin d’améliorer la personnalisation des LLM et la pertinence de leurs réponses.

- Explorer l’approche modulaire de LangChain, qui permet d’ajuster dynamiquement les prompts et les modèles sans nécessiter de modifications importantes du code.

- Améliorer les applications basées sur les LLM en intégrant des techniques de génération augmentée par récupération (RAG) avec LangChain. Vous apprendrez comment l’intégration du RAG permet d’obtenir une meilleure précision et des réponses plus contextualisées.


## 2. Librairies

In [1]:

def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## 3. Langchain

## 3.1 Le modèle

Le modèle est un LLM. Il prend en entrée du texte, des images ... et donne en sortie généralement du texte. Le LLM couplé avec langchain est un outil puissant qui permet de faire le développement facile et rapide des applications.

In [23]:
llm = ChatOpenAI(
    model=  "gpt-5-mini", #"gpt-4.1-mini",
    temperature=0.5,
    #top_p=0.2,
    #top_k=1,
    max_tokens=256,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [24]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

In [4]:
prompt = "Paris is"
response = llm.invoke(prompt)

print(f"prompt: {prompt}\n")
print(f"response : {response.content}\n")

prompt: Paris is

response : Paris is the capital city of France. It is known for its rich history, iconic landmarks such as the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, and its vibrant culture, art, fashion, and cuisine. Paris is often referred to as "The City of Light" and is one of the most popular tourist destinations in the world. If you want to know more specific information about Paris, feel free to ask!



### 3.2 Chat message

Le modèle de chat prend en entrée une liste de messages et renvoie un nouveau message. Tous les messages possèdent une propriété `role` et une propriété `content`. Voici les types de messages les plus couramment utilisés :

- `SystemMessage` : Utilisez ce type de message pour définir ou orienter le comportement de l’IA. Ce message est généralement transmis en premier dans une séquence de messages en entrée.
- `HumanMessage` : Ce type de message représente une intervention d’un utilisateur interagissant avec le modèle de chat.
- `AIMessage` : Ce type de message, qui peut être du texte ou une demande pour invoquer un outil, représente une réponse générée par le modèle de chat.

Vous pouvez trouver plus de types de messages ici : *LangChain built-in message types* (https://python.langchain.com/v0.2/docs/how_to/custom_chat_model/#messages).


In [5]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [8]:
msg = llm.invoke(
    [
        SystemMessage(content="You are a helpful AI bot that assits tourist in Cameroon. Help them choose the best place to visit in Cameroon regarding a city."),
        HumanMessage(content="Hi, i'm actually in Limbé. Which place should I visit?")
    ]
)
print(msg.content)

Hello! Since you're in Limbé, you're in a fantastic location with some great places to visit. Here are some top recommendations:

1. **Limbe Botanical Garden** – A beautiful garden showcasing a variety of tropical plants and a great spot for nature lovers.

2. **Limbe Wildlife Centre** – A sanctuary for rescued primates and other animals, perfect for wildlife enthusiasts.

3. **Limbe Beach** – Enjoy the scenic Atlantic coastline, relax on the beach, or try some local seafood at nearby restaurants.

4. **Mount Cameroon** – If you're up for an adventure, consider hiking Mount Cameroon, the highest peak in West Africa, located not far from Limbé.

5. **Fako Mountain** – Explore the surrounding volcanic landscapes and enjoy hiking trails with stunning views.

Would you like recommendations for restaurants or cultural sites as well?


In [9]:
msg = llm.invoke(
    [
        SystemMessage(content="You are a supportive AI bot that suggests fitness activities to a user in one short sentence"),
        HumanMessage(content="I like high-intensity workouts, what should I do?"),
        AIMessage(content="You should try a CrossFit class"),
        HumanMessage(content="How often should I attend?")
    ]
)
print(msg.content)

Aim for 3-4 high-intensity workouts per week to maximize results and allow recovery.


### 3.3 Prompt templates

Les modèles de prompts (prompt templates) aident à transformer les entrées utilisateur et les paramètres en instructions destinées à un modèle de langage. Vous pouvez utiliser ces modèles pour guider la réponse du modèle, l’aider à comprendre le contexte et générer une sortie pertinente et cohérente basée sur le langage.

Ensuite, explorez plusieurs types de modèles de prompts différents.


#### Les Prompt templates textuels

In [6]:
from langchain_core.prompts import PromptTemplate

In [13]:
query = "Tell me a {adjective} joke about {field}."
prompt = PromptTemplate.from_template(query)
variables = {
    "adjective": "funny",
    "field": "AI"
}

In [14]:
prompt.invoke(variables)

StringPromptValue(text='Tell me a funny joke about AI.')

In [15]:
prompt.format(adjective="funny", field="AI")

'Tell me a funny joke about AI.'

#### Chat prompt templates

Ces templates contiennent des listes de templates. Ils permettent de formater une liste de messages.

In [7]:
from langchain_core.prompts import ChatPromptTemplate



In [16]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI bot that assits tourist in Cameroon. Help them choose the best place to visit in Cameroon regarding a city."),
        ("user", "I'm in {city}. What is your advice?")
    ]
)
inputs = {"city": "Douala"}
#prompt.invoke(input_)
prompt.format(city="Douala")

"System: You are a helpful AI bot that assits tourist in Cameroon. Help them choose the best place to visit in Cameroon regarding a city.\nHuman: I'm in Douala. What is your advice?"

In [17]:
chain = prompt | llm
response = chain.invoke(input = inputs)
print(response.content)

Welcome to Douala! As the economic capital of Cameroon, Douala is a vibrant city with a mix of modernity and culture. Here are some recommendations for places to visit and things to do in and around Douala:

1. **Douala Central Market (Marché Central)**  
   Experience the bustling atmosphere of one of the largest markets in the city. You can find local crafts, fabrics, spices, and fresh produce here.

2. **La Nouvelle Liberté Statue**  
   This iconic sculpture made from recycled materials is a symbol of Douala’s creativity and resilience. It’s a great spot for photos and to appreciate local art.

3. **Douala Maritime Museum**  
   Learn about the maritime history of Douala and Cameroon’s coastal heritage.

4. **Bonanjo District**  
   Explore the administrative and business heart of the city with colonial architecture and some nice cafes.

5. **Akwa District**  
   Known for its lively nightlife, restaurants, and shopping centers.

6. **Littoral Beaches**  
   While Douala itself is 

#### Les formats de sortie (Output parsers)

Les formats de sortie (output parsers) prennent la sortie d’un LLM et la transforment dans un format plus approprié. L’analyse de la sortie est très utile lorsque vous utilisez des LLM pour générer une forme quelconque de données structurées, ou pour normaliser la sortie provenant des modèles de chat et d’autres LLM.

LangChain propose de nombreux types d’analyseurs de sortie. Voici une [liste](https://python.langchain.com/v0.2/docs/concepts/#output-parsers) des analyseurs de sortie pris en charge par LangChain. Dans ce TP, vous utiliserez :

- `JSON` : Renvoie un objet JSON tel que spécifié. Vous pouvez définir un modèle Pydantic et il renverra un JSON correspondant à ce modèle. C’est probablement l’analyseur de sortie le plus fiable pour obtenir des données structurées sans utiliser l’appel de fonctions.
- `CSV` : Renvoie une liste de valeurs séparées par des virgules.


##### JSON PARSER

In [11]:
class city_description(BaseModel):
    city_name: str=Field(description="The name of the city")
    touristic_site_name: str=Field(description="The name of toursitic site to visit")
    free: bool=Field(description="If the entrance is free or not")

output_parser = JsonOutputParser(pydantic_object=city_description)
format_instructions = output_parser.get_format_instructions()

template="""You are a helpful AI bot that assits tourist in Cameroon.

Task: Help them choose the 4 best places to visit in {city} and cities near in Json format.

{format_instructions}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["city"],  
    partial_variables={"format_instructions": format_instructions},  
)

In [12]:
chain = prompt | llm | output_parser

In [13]:
result = chain.invoke('Yaoundé')
result

[{'city_name': 'Yaoundé',
  'touristic_site_name': 'Mvog-Betsi Zoo',
  'free': False},
 {'city_name': 'Yaoundé',
  'touristic_site_name': 'National Museum of Yaoundé',
  'free': False},
 {'city_name': 'Yaoundé',
  'touristic_site_name': 'Benedictine Museum',
  'free': False},
 {'city_name': 'Mbalmayo',
  'touristic_site_name': 'Lobéké National Park',
  'free': False}]

In [16]:
tmp = {'city_name': 'Yaoundé',
  'touristic_site_name': 'Mvog-Betsi Zoo',
  'free': False}
data = []
data.append(tmp)

In [17]:
data

[{'city_name': 'Yaoundé',
  'touristic_site_name': 'Mvog-Betsi Zoo',
  'free': False}]

In [14]:
type(result)

list

In [30]:
import json

In [31]:
json.dumps(result)

'[{"city_name": "Yaound\\u00e9", "touristic_site_name": "Mvog-Betsi Zoo", "free": false}, {"city_name": "Yaound\\u00e9", "touristic_site_name": "National Museum of Yaound\\u00e9", "free": false}, {"city_name": "Yaound\\u00e9", "touristic_site_name": "Benedictine Museum", "free": true}, {"city_name": "Mbalmayo", "touristic_site_name": "Lob\\u00e9k\\u00e9 National Park", "free": false}]'

In [8]:

from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

In [9]:
class Joke(BaseModel):
    setup: str=Field(description="question to set up a joke")
    punchline: str=Field(description="answer to resolve the joke")

In [10]:
joke_query = "Tell me a joke."
output_parser = JsonOutputParser(pydantic_object=Joke)
format_instructions = output_parser.get_format_instructions()

In [13]:
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],  # Dynamic variables that will be provided when invoking the chain
    partial_variables={"format_instructions": format_instructions},  # Static variables set once when creating the prompt
)

In [14]:
chain = prompt | llm | output_parser

In [15]:
chain.invoke({"query": joke_query})

{'setup': "Why don't scientists trust atoms?",
 'punchline': 'Because they make up everything!'}

#### CSV Parser

In [16]:
from langchain.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()
format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="Answer the user query. {format_instructions}\n List five {subject}.",
    input_variables=["suject"],
    partial_variables={"format_instructions": format_instructions},
)

In [17]:
chain = prompt | llm | output_parser
chain.invoke({"subject": "ice cream flavors"})

['vanilla',
 'chocolate',
 'strawberry',
 'mint chocolate chip',
 'cookies and cream']

In [18]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

# Create your JSON parser
class movie(BaseModel):
    title: str=Field(description="movie title")
    director: str=Field(description="director name")
    year: int=Field(description="year of publication")
    genre: str=Field(description="The movie genre")
json_parser = JsonOutputParser(pydantic_object=movie)
format_instructions = json_parser.get_format_instructions()


# Create the format instructions
# format_instructions = """RESPONSE FORMAT: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
# {
#   "title": "movie title",
#   "director": "director name",
#   "year": 2000,
#   "genre": "movie genre"
# }

# IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON."""

# Create prompt template with instructions
prompt_template = PromptTemplate(
    template="""You are a JSON-only assistant.

Task: Generate info about the movie "{movie_name}" in JSON format.

{format_instructions}
""",
    input_variables=["movie_name"],
    partial_variables={"format_instructions": format_instructions},
)

# Create the chain
movie_chain = prompt_template | llm | json_parser

# Test with a movie name
movie_name = "The Matrix"
result = movie_chain.invoke({"movie_name": movie_name})



In [20]:
result

{'title': 'The Matrix',
 'director': 'Lana Wachowski, Lilly Wachowski',
 'year': 1999,
 'genre': 'Science Fiction'}

In [19]:

print("Parsed result:")
print(f"Title: {result['title']}")
print(f"Director: {result['director']}")
print(f"Year: {result['year']}")
print(f"Genre: {result['genre']}")

Parsed result:
Title: The Matrix
Director: Lana Wachowski, Lilly Wachowski
Year: 1999
Genre: Science Fiction


### Documents loaders

#### pdf loader

In [23]:

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")

document = loader.load()

In [24]:
document

[Document(metadata={'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'page': 0}, page_content="* corresponding author - jkim72@kent.edu \nRevolutionizing Mental Health Care through \nLangChain: A Journey with a Large Language \nModel\nAditi Singh \n Computer Science  \n Cleveland State University  \n a.singh22@csuohio.edu \nAbul Ehtesham  \nThe Davey Tree Expert \nCompany  \nabul.ehtesham@davey.com \nSaifuddin Mahmud  \nComputer Science & \nInformation Systems  \n Bradley University  \nsmahmud@bradley.edu  \nJong-Hoon Kim* \n Computer Science,  \nKent State University,  \njkim72@kent.edu \nAbstract— Mental health challenges are on the rise in our \nmodern society, and the imperative to address mental disorders, \nespecially regarding anxiety, depression, and suicidal thoughts, \nunderscores the need for effective interventions. This paper \ndelves into the application of recent advancements in pretrained \ncontex

In [25]:
print(document[1].page_content[:1000])

LangChain helps us to unlock the ability to harness the 
LLM’s immense potential in tasks such as document analysis, 
chatbot development, code analysis, and countless other 
applications. Whether your desire is to unlock deeper natural 
language understanding , enhance data, or circumvent 
language barriers through translation, LangChain is ready to 
provide the tools and programming support you need to do 
without it that it is not only difficult but also fresh for you. Its 
core functionalities encompass: 
1. Context-Aware Capabilities: LangChain facilitates the 
development of applications that are inherently 
context-aware. This means that these applications can 
connect to a language model and draw from various 
sources of context, such as prompt instructions, a few-
shot examples, or existing content, to ground their 
responses effectively. 
2. Reasoning Abilities: LangChain equips applications 
with the capacity to reason effectively. By relying on a 
language model, these appl

##### **URL et website loader**


In [27]:

from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.bible.com/bible/1854/MAT.1.NTE12")


web_data = loader.load()


print(web_data[0].page_content[:1000])

Mt 1 | NTE12 Bible | YouVersionYouVersion LogoBiblePlansVideosSearchGet the appLanguage SelectorSearch IconMt 1NTE12ParallelREADER SETTINGSMt 1


1Nlaṅ abonde nda-bod Yesus 1Kalara#1,1-17 Kalara mvòṅ bod Yesus: Nlaṅ te wayi na walëdë na e mëtiṅ ya Zamba angalig Abraham ai David matoban a nyòl Yesus. mvòṅ bod Yesus Kristus, man David, man Abraham. 2Abraham abye Isaak, Isaak ny'abye Yakòb, Yakòb abye Yuda ai babënyaṅ, 3Yuda abye Farès ban Zara, ai Tamar, Farès abye Esrom, Esrom abye Aram, 4Aram abye Aminadab, Aminadab abye Naason, Naason abye Salmon, 5Salmon abye Boaz ai Rahab, Boaz abye Yobèd ai Ruth, Yobèd abye Yesse, 6Yesse nyè abye nkukuma David. David abye Salomon ai mininga Uria, 7Salomon abye Roboam, Roboam abye Abia, Abia abye Asaf. 8Asaf abye Yosafat, Yosafat abye Yoram, Yoram abye Ozia, 9Ozia abye Yoatam, Yoatam abye Akas, Akas abye Ezekias, 10Ezekias abye Manasse, Manasse abye Amos, Amos abye Yosia, 11Yosia abye Yekonia ai babënyaṅ abog mëkaban a Babilon. 12A mvus mëkaban ya B

In [28]:
web_data

[Document(metadata={'source': 'https://www.bible.com/bible/1854/MAT.1.NTE12', 'title': 'Mt 1 | NTE12 Bible | YouVersion', 'description': 'Nla&#7749; abonde nda-bod Yesus  Kalara#,1-17 Kalara mv&#242;&#7749; bod Yesus: Nla&#7749; te wayi na wal&#235;d&#235; na e m&#235;ti&#7749; ya Zamba angalig Abraham ai David matoban a ny&#242;l Yesus', 'language': 'en'}, page_content="Mt 1 | NTE12 Bible | YouVersionYouVersion LogoBiblePlansVideosSearchGet the appLanguage SelectorSearch IconMt 1NTE12ParallelREADER SETTINGSMt 1\n\n\n1Nlaṅ abonde nda-bod Yesus 1Kalara#1,1-17 Kalara mvòṅ bod Yesus: Nlaṅ te wayi na walëdë na e mëtiṅ ya Zamba angalig Abraham ai David matoban a nyòl Yesus. mvòṅ bod Yesus Kristus, man David, man Abraham. 2Abraham abye Isaak, Isaak ny'abye Yakòb, Yakòb abye Yuda ai babënyaṅ, 3Yuda abye Farès ban Zara, ai Tamar, Farès abye Esrom, Esrom abye Aram, 4Aram abye Aminadab, Aminadab abye Naason, Naason abye Salmon, 5Salmon abye Boaz ai Rahab, Boaz abye Yobèd ai Ruth, Yobèd abye Yess

#### Text splitter

In [29]:

from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")

chunks = text_splitter.split_documents(document)

print(len(chunks))

147


In [31]:
chunks[5].page_content

'individuals seeking guidance and support in these critical areas. \nMindGuide lever ages the capabilities of LangChain and its \nChatModels, specifically Chat OpenAI, as the bedrock of its'

In [33]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter

# Load the LangChain paper
paper_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf"
pdf_loader = PyPDFLoader(paper_url)
pdf_document = pdf_loader.load()

# Load content from LangChain website
web_url = "https://python.langchain.com/v0.2/docs/introduction/"
web_loader = WebBaseLoader(web_url)
web_document = web_loader.load()

# Create two different text splitters
splitter_1 = CharacterTextSplitter(chunk_size=300, chunk_overlap=30, separator="\n")
splitter_2 = CharacterTextSplitter(chunk_size=500, chunk_overlap=10,separator="\n")

# Apply both splitters to the PDF document
chunks_1 = splitter_1.split_documents(pdf_document)
chunks_2 = splitter_2.split_documents(pdf_document)

# Define a function to display document statistics
def display_document_stats(docs, name):
    """Display statistics about a list of document chunks"""
    total_chunks = len(docs)
    total_chars = sum(len(doc.page_content) for doc in docs)
    avg_chunk_size = total_chars / total_chunks if total_chunks > 0 else 0
    
    # Count unique metadata keys across all documents
    all_metadata_keys = set()
    for doc in docs:
        all_metadata_keys.update(doc.metadata.keys())
    
    # Print the statistics
    print(f"\n=== {name} Statistics ===")
    print(f"Total number of chunks: {total_chunks}")
    print(f"Average chunk size: {avg_chunk_size:.2f} characters")
    print(f"Metadata keys preserved: {', '.join(all_metadata_keys)}")
    
    if docs:
        print("\nExample chunk:")
        example_doc = docs[min(5, total_chunks-1)]  # Get the 5th chunk or the last one if fewer
        print(f"Content (first 150 chars): {example_doc.page_content[:150]}...")
        print(f"Metadata: {example_doc.metadata}")
        
        # Calculate length distribution
        lengths = [len(doc.page_content) for doc in docs]
        min_len = min(lengths)
        max_len = max(lengths)
        print(f"Min chunk size: {min_len} characters")
        print(f"Max chunk size: {max_len} characters")

# Display stats for both chunk sets
display_document_stats(chunks_1, "Splitter 1")
display_document_stats(chunks_2, "Splitter 2")


=== Splitter 1 Statistics ===
Total number of chunks: 95
Average chunk size: 263.73 characters
Metadata keys preserved: page, source

Example chunk:
Content (first 150 chars): comprehensive support within the field of mental health. 
Additionally, the paper discusses the implementation of 
Streamlit to enhance the user ex pe...
Metadata: {'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf', 'page': 0}
Min chunk size: 49 characters
Max chunk size: 299 characters

=== Splitter 2 Statistics ===
Total number of chunks: 56
Average chunk size: 448.11 characters
Metadata keys preserved: page, source

Example chunk:
Content (first 150 chars): with severe intellectual disorders do no longer have get entry 
to the necessary remedy they require. This remedy gap 
intensifies the weight of intel...
Metadata: {'source': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf'